<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/query-api/locate-filings-urls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Find URLs of SEC Filings Over an Extended Period

A common use case for the Query API is retrieving filing URLs and accession numbers from a large range of filings over an extended disclosure period, and saving the metadata locally for further processing.

Use cases for such data include:
- Downloading filings via the [Filing Download API](https://sec-api.io/docs/sec-filings-render-api)
- Generating PDF versions of filings and exhibits using the [PDF Generator API](https://sec-api.io/docs/sec-filings-render-api#pdf-generator-api)
- Extracting content sections with the [Extractor API](https://sec-api.io/docs/sec-filings-item-extraction-api)
- Accessing structured XBRL data in JSON format from filings with the [XBRL-JSON API](https://sec-api.io/docs/xbrl-to-json-converter-api)
- And more

One such example is to locate all Form 10-Q filings submitted to the SEC between 2014 and 2023, and save the URLs pointing to the HTML version of each filing to a local file. The initial search query is:

```
formType:"10-Q" AND filedAt:[2014-01-01 TO 2023-12-31]
```

By default, the Query API can return a maximum of 10,000 filings per search query. Since the API returns 50 filings per request, pagination is necessary to retrieve more than 50 results. The `from` parameter is used to paginate through the results, specifying the starting position for each batch, and is incremented by 50 with each subsequent API call.

However, because there are typically three 10-Q filings per company each quarter, and more than 5,000 active companies per year, this query will return far more than 10,000 filings, meaning pagination alone is insufficient. To retrieve all filings, the query needs to be broken down into smaller chunks, such as by year and month, with pagination applied to each chunk. Since no single month contains more than 10,000 Form 10-Q filings, pagination within each month ensures complete retrieval.

For example, the search queries for 10-Q filings filed in January and February of 2014 would look like this:

```
# Search for 10-Q filings filed in January 2014
formType:"10-Q" AND filedAt:[2014-01-01 TO 2014-01-31]

# Search for 10-Q filings filed in February 2014
formType:"10-Q" AND filedAt:[2014-02-01 TO 2014-02-28]
```

The following approach iterates over the years from 2014 to 2023, and for each year, loops through the months from January to December. For each month, multiple Query API requests are executed by incrementing the `from` parameter in steps of 50 to capture all 10-Q filings for that month. The URLs of the filings are then saved to a file on the local disk.

The same approach can be applied to other filing types and filtering criteria, such as Form 10-K filings, Form 8-K filings, filings by specific companies, and more.

In [ ]:
pip install sec-api

In [ ]:
from sec_api import QueryApi

queryApi = QueryApi(api_key="YOUR_API_KEY")

In [ ]:
search_parameters = {
  "query": "<PLACEHOLDER>", # will be set during runtime
  "from": "0", # will be incremented by 50 during runtime
  "size": "50",
  "sort": [{ "filedAt": { "order": "desc" } }]
}

In [ ]:
# open the file to store the filing URLs
log_file = open("filing_urls.csv", "a")

# fetch filings filed in 2023, then 2022, 2021, ... up to 2014
# uncomment line below to fetch all filings filed between 2014 and 2023
# for year in range(2023, 2013, -1):
for year in range(2023, 2022, -1):
    print(f"Starting search for: {year}")

    # iterate over all months in a year and
    # build a query to fetch all 10-Q filings
    # filed in that month and year
    for month in range(1, 13, 1):
        search_parameters["from"] = 0

        form_type_query = 'formType:"10-Q"'
        date_range_query = f"filedAt:[{year}-{month:02d}-01 TO {year}-{month:02d}-31]"
        search_parameters["query"] = form_type_query + " AND " + date_range_query

        print("Starting filing search for: ", search_parameters["query"])

        # paginate through results by increasing "from" parameter
        # until all results are fecthed and no more filings are returned
        # uncomment line below to fetch all 10,000 filings per month
        # for from_param in range(0, 9950, 50):
        for from_param in range(0, 50, 50):
            search_parameters["from"] = from_param

            response = queryApi.get_filings(search_parameters)

            # stop if no more filings are returned
            if len(response["filings"]) == 0:
                break

            # for each filing, get the URL of the filing
            # set in the dict key "linkToFilingDetails"
            urls_list = list(
                map(lambda x: x["linkToFilingDetails"], response["filings"])
            )

            # transform the list of URLs into a single string by
            # joining all list elements, add a new-line character between each element
            urls_string = "\n".join(urls_list) + "\n"

            log_file.write(urls_string)

log_file.close()

Starting search for: 2023
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-01-01 TO 2023-01-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-02-01 TO 2023-02-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-03-01 TO 2023-03-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-04-01 TO 2023-04-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-05-01 TO 2023-05-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-06-01 TO 2023-06-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-07-01 TO 2023-07-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-08-01 TO 2023-08-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-09-01 TO 2023-09-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-10-01 TO 2023-10-31]
Starting filing search for:  formType:"10-Q" AND filedAt:[2023-11-01 TO 2023-11-31]
Starting filing search for:  formType:"10-Q" AND f

In [ ]:
# let's inspect the content of file
log_file = open("filing_urls.csv", "r")
urls = log_file.readlines()
# replace \n at the end of each line with empty string
urls = list(map(lambda x: x.replace("\n", ""), urls))
print("Number of URLs fetched:\n", len(urls))
print("Snippet of 10-Q filing URLs fetched: ")
urls[:10]

Number of URLs fetched:
 600
Snippet of 10-Q filing URLs fetched: 


['https://www.sec.gov/Archives/edgar/data/1596993/000159699323000011/lpg-20221231x10q.htm',
 'https://www.sec.gov/Archives/edgar/data/1666138/000166613823000032/atkr-20221230.htm',
 'https://www.sec.gov/Archives/edgar/data/96021/000009602123000033/syy-20221231.htm',
 'https://www.sec.gov/Archives/edgar/data/50725/000005072523000006/gff-20221231.htm',
 'https://www.sec.gov/Archives/edgar/data/17313/000001731323000006/cswc-20221231.htm',
 'https://www.sec.gov/Archives/edgar/data/1169561/000116956123000013/cvlt-20221231.htm',
 'https://www.sec.gov/Archives/edgar/data/1680873/000168087323000008/hffg-20220930.htm',
 'https://www.sec.gov/Archives/edgar/data/1680873/000168087323000007/hffg-20220630.htm',
 'https://www.sec.gov/Archives/edgar/data/1680873/000168087323000006/hffg-20220331.htm',
 'https://www.sec.gov/Archives/edgar/data/1681622/000168162223000013/var-20221230.htm']